# Extended Kalman Filter, A Revisit on the Non-Linear State Space Model

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from bunobee.models.ssp.kalman_1d_ekf import (
    kalman_dk_smoother_1d_ekf,
    kalman_filter_1d_ekf,
    kalman_rts_smoother_1d_ekf,
 )
from bunobee.models.ssp import plot_states, transform_to_ekf
from bunobee.models.ssp.utils import construct_states_prior

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# False → plain Kalman filter; True → enforce a_obs at sampled windows
USE_DISCLOSURE = True 
# number of disclosure windows to sample
N_PERIODS      = 3    
# consecutive steps per window
N_POINTS       = 10     
# disclose noise variance
DISC_OBS_SIGMA = 0.01

# for EKF positivity constraint
EXPONENT = 0.5


In [ ]:
df = pd.read_csv(
    '../resource/simple/df.csv', parse_dates=['date']
)
# for quick testing
df = df[-365:].reset_index(drop=True)

saturation_df = pd.read_csv(
    '../resource/simple/saturation_df.csv',
)
coefs_df = pd.read_csv(
    '../resource/simple/coefs_df.csv',
)

regressors = saturation_df["regressor"].tolist()
response = df["sales"].values

response_norm = response.mean()
y = np.log(response / response_norm)
y = jnp.array(y)
X = df[regressors].values
sat_arr = saturation_df.set_index("regressor").loc[regressors, "saturation"].values
X = np.log1p(X / sat_arr)

Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))
positivity_idx = positivity_idx.astype(bool)

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df["date"], y, label="y")

In [ ]:
n_steps = y.shape[0]
n_states = Z.shape[-1]
state_labels = ["intercept"] + regressors
ssp_priors_nat = None

if coefs_df is not None:
    coefs_lookup = coefs_df.set_index("regressor")["coef"]
    # natural-scale true state: intercept linear, channel coefficients in λ-space
    true_state_nat = jnp.array([0.0] + [coefs_lookup[r] for r in regressors])

    if USE_DISCLOSURE:
        ssp_priors_nat = construct_states_prior(
            n_steps=n_steps,
            n_states=n_states,
            true_states=true_state_nat,
            regressors=regressors,
            n_periods=N_PERIODS,
            n_points=N_POINTS,
            obs_scale=DISC_OBS_SIGMA,
        )
        # keep the `_nat` suffix v3 uses for natural-scale obs vars
        ssp_priors_nat = ssp_priors_nat.rename({"a_obs": "a_obs_nat", "P_obs": "P_obs_nat"})
        print("a_obs_nat shape:", ssp_priors_nat["a_obs_nat"].shape)
        print("P_obs_nat shape:", ssp_priors_nat["P_obs_nat"].shape)
        print("n disclosed steps:", ssp_priors_nat.sizes["obs_point"])
    else:
        print("disclosure disabled")
else:
    print("no ground truth state available; disclosure disabled")


In [ ]:
import xarray as xr

n_regressors = X.shape[-1]
positivity_array = np.array([False] + [True] * n_regressors)

base_ds_nat = xr.Dataset(
    {
        "a0_nat":                  (("state",),                np.array([0.0] + [0.1] * n_regressors)),
        # Tighter natural-scale variance on channels: lognormal moment-match blows
        # up Var/μ² ratios, so shrinking 0.1 → 0.01 keeps the a-space prior compact.
        "P0_nat":                  (("state", "state_dual"),   np.diag(np.array([1.0] + [0.01] * n_regressors))),
        # (level, regressor) process-noise scale priors in the natural scale.
        # sigma_q is a standard deviation, not a latent-state mean/variance pair.
        "sigma_q_loc_prior_nat":   (("sigma_q_group",),        np.array([0.05, 0.01])),
        "sigma_q_scale_prior_nat": (("sigma_q_group",),        np.array([0.05, 0.01])),
        # Truncation bounds in natural scale. For positive states transform_to_ekf
        # converts sigma_q with the same local positive scale map used for the
        # TruncatedNormal prior parameters; it does not moment-match sigma_q like a0/P0.
        "sigma_q_low_prior_nat":   (("sigma_q_group",),        np.array([1e-5, 1e-5])),
        "sigma_q_high_prior_nat":  (("sigma_q_group",),        np.array([0.1, 0.1])),
        "positivity_idx":          (("state",),                positivity_array),
    },
    coords={
        "state":         state_labels,
        "state_dual":    state_labels,
        "sigma_q_group": ["level", "regressor"],
    },
)

if ssp_priors_nat is not None:
    ssp_priors_nat = xr.merge([base_ds_nat, ssp_priors_nat])
else:
    null_disc = xr.Dataset(
        {
            "a_obs_nat":      (("time", "state"), np.zeros((n_steps, n_states))),
            "P_obs_nat":      (("time", "state"), np.full((n_steps, n_states), np.inf)),
            "obs_idx":        (("obs_point",),    np.array([], dtype=int)),
        },
        coords={
            "time":      np.arange(n_steps),
            "obs_point": np.array([], dtype=int),
        },
    )
    ssp_priors_nat = xr.merge([base_ds_nat, null_disc])

print("----- Natural-scale priors -----")
print(ssp_priors_nat)

ssp_priors_ekf = transform_to_ekf(ssp_priors_nat, exponent=EXPONENT)

print("\n----- EKF-space priors -----")
print(ssp_priors_ekf)
print("a0:", ssp_priors_ekf["a0"].values)
print("P0 diag:", np.diag(ssp_priors_ekf["P0"].values))
print("sigma_q loc / scale:",
      ssp_priors_ekf["sigma_q_loc_prior"].values, "/",
      ssp_priors_ekf["sigma_q_scale_prior"].values)
print("sigma_q low / high:",
      ssp_priors_ekf["sigma_q_low_prior"].values, "/",
      ssp_priors_ekf["sigma_q_high_prior"].values)
ssp_priors_ekf

In [ ]:
# def test_run(ssp_priors, exponent=EXPONENT):
#     a0      = jnp.array(ssp_priors_ekf["a0"].values)
#     # EKF expects diagonal variances; collapse the full a-space covariance.
#     P0      = jnp.diag(jnp.array(ssp_priors_ekf["P0"].values))
#     a_obs   = jnp.array(ssp_priors_ekf["a_obs"].values)
#     P_obs   = jnp.array(ssp_priors_ekf["P_obs"].values)
#     obs_idx = np.asarray(ssp_priors_ekf["obs_idx"].values)

#     print("a0 shape:", a0.shape)
#     print(f"  a0[0] = {a0[0]:.3f}  (intercept, linear space)")
#     print(
#         f"  a0[1] = {a0[1]:.3f}  (channel, a-space → λ₀ = exp({EXPONENT}·a0) = "
#         f"{jnp.exp(EXPONENT * a0[1]):.3f})"
#     )
#     print("P0:", P0)

#     sigma_h = jnp.array(0.1)
#     # using prior loc as a test value
#     sigma_q_raw = jnp.array(ssp_priors["sigma_q_loc_prior"].values)
#     sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
#     print("sigma_q expanded:", sigma_q.shape, sigma_q)

#     lp, at, Pt, _, _, _ = kalman_filter_1d_ekf(
#         a0=a0,
#         P0=P0,
#         sigma_h=sigma_h,
#         sigma_q=sigma_q,
#         y=y,
#         Z=Z,
#         logp=True,
#         exponent=EXPONENT,
#         positivity_idx=positivity_idx,
#     )

#     # recover intensities: intercept stays as-is, channels via exp(EXPONENT·a)
#     lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

#     print("lp:", lp)
#     print("at shape:", at.shape)
#     print("intercept at[-1,0]:", at[-1, 0], "  ← linear space")
#     print(f"channel lam[-1,1]: {lam[-1, 1]}  ← λ = exp({exponent}·a)")

# test_run(ssp_priors_ekf, positivity_idx)


In [ ]:
def make_nuts_fn(ssp_priors_ekf, y, Z, positivity_idx, exponent):
    a0    = jnp.array(ssp_priors_ekf["a0"].values)
    P0    = jnp.diag(jnp.array(ssp_priors_ekf["P0"].values))
    a_obs = jnp.array(ssp_priors_ekf["a_obs"].values)
    P_obs = jnp.array(ssp_priors_ekf["P_obs"].values)
    # NumPyro samples sigma_q in the (level, shared-regressor) parameterisation;
    # collapse the per-state hyperpriors to that two-element form.
    sigma_q_loc_prior   = jnp.array(ssp_priors_ekf["sigma_q_loc_prior"].values)
    sigma_q_scale_prior = jnp.array(ssp_priors_ekf["sigma_q_scale_prior"].values)
    sigma_q_low_prior   = jnp.array(ssp_priors_ekf["sigma_q_low_prior"].values)
    sigma_q_high_prior  = jnp.array(ssp_priors_ekf["sigma_q_high_prior"].values)
    positivity_idx      = positivity_idx.astype(bool)

    def nuts_fn():
        sigma_h = numpyro.sample(
            "sigma_h",
            dist.TruncatedNormal(0.1, 1.0, high=1.0, low=1e-5),
        )
        sigma_q_raw = numpyro.sample(
            "sigma_q",
            dist.TruncatedNormal(
                sigma_q_loc_prior,
                sigma_q_scale_prior,
                low=sigma_q_low_prior,
                high=sigma_q_high_prior,
            ),
        )
        n_regressors = Z.shape[-1] - 1
        sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

        lp, at, Pt, vt, Ft, Kt = kalman_filter_1d_ekf(
            a0=a0,
            P0=P0,
            sigma_h=sigma_h,
            sigma_q=sigma_q,
            y=y,
            Z=Z,
            logp=True,
            exponent=exponent,
            positivity_idx=positivity_idx,
            a_obs=a_obs,
            P_obs=P_obs,
        )
        lam = jnp.where(positivity_idx, jnp.exp(jnp.clip(exponent * at, -10.0, 10.0)), at)

        numpyro.factor("lp", lp)
        numpyro.deterministic("at", at)
        numpyro.deterministic("Pt", Pt)
        numpyro.deterministic("lam", lam)
        numpyro.deterministic("mu", jnp.sum(Z * lam, -1))
        numpyro.deterministic("sigma_q_full", sigma_q)

    return nuts_fn


In [ ]:
nuts_fn = make_nuts_fn(ssp_priors_ekf, y, Z, positivity_idx, exponent=EXPONENT)
nuts_kernel = NUTS(nuts_fn)
mcmc = MCMC(
    nuts_kernel,
    num_warmup=400,
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key)

In [ ]:
import arviz as az
from bunobee.models.ssp.utils import posterior_to_xarray

# Group by chain so every site has leading (chain, draw) axes — matches v2's pattern
# and is required by posterior_to_xarray.
numpyro_posteriors = mcmc.get_samples(group_by_chain=True)

# ── Post-MCMC RTS smoother ────────────────────────────────────────────────
# RTS only needs (at, Pt, sigma_q_full); all three are already deterministics on
# the MCMC posterior, so no filter rerun is required.
at_draws      = numpyro_posteriors["at"]            # (n_chains, n_draws, T, n_states)
Pt_draws      = numpyro_posteriors["Pt"]
sigma_q_draws = numpyro_posteriors["sigma_q_full"]  # (n_chains, n_draws, n_states)

at_smooth, Pt_smooth = jax.vmap(jax.vmap(
    lambda at_i, Pt_i, sigma_q_i: kalman_rts_smoother_1d_ekf(
        at=at_i,
        Pt=Pt_i,
        sigma_q=sigma_q_i,
    )
))(at_draws, Pt_draws, sigma_q_draws)

lam_smooth = jnp.where(
    positivity_idx[None, None, None, :],
    jnp.exp(jnp.clip(EXPONENT * at_smooth, -10.0, 10.0)),
    at_smooth,
)
mu = jnp.einsum("cdts,ts->cdt", lam_smooth, Z)

print(f"RTS at_smooth shape: {at_smooth.shape}")

# Smoother results flow straight into xarray; caller owns the dim/coord schema.
posterior = posterior_to_xarray(
    {
        **numpyro_posteriors,
        "at_smooth":  np.asarray(at_smooth),
        "Pt_smooth":  np.asarray(Pt_smooth),
        "lam_smooth": np.asarray(lam_smooth),
        "mu":         np.asarray(mu),
    },
    dims={
        "sigma_q":      ["sigma_q_group"],
        "sigma_q_full": ["state"],
        "at":           ["time", "state"],
        "Pt":           ["time", "state"],
        "lam":          ["time", "state"],
        "at_smooth":    ["time", "state"],
        "Pt_smooth":    ["time", "state"],
        "lam_smooth":   ["time", "state"],
        "mu":           ["time"],
    },
    coords={
        "state":         state_labels,
        "sigma_q_group": ["level", "regressor"],
        "time":          df["date"].values,
    },
)

idata = az.InferenceData(posterior=posterior)
az.summary(idata, var_names=["sigma_h", "sigma_q"])


In [ ]:
a_obs_plot   = np.asarray(ssp_priors_nat["a_obs_nat"].values) if USE_DISCLOSURE else None
P_obs_plot   = np.asarray(ssp_priors_nat["P_obs_nat"].values) if USE_DISCLOSURE else None
obs_idx_plot = np.asarray(ssp_priors_nat["obs_idx"].values)   if USE_DISCLOSURE else None

plot_states(
    posterior,
    df["date"].values,
    state_labels,
    states_key="lam",
    coefs_df=coefs_df,
    obs_idx=obs_idx_plot,
    a_obs=a_obs_plot,
    P_obs=P_obs_plot,
    title=(
        f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()

plot_states(
    posterior,
    df["date"].values,
    state_labels,
    states_key="lam_smooth",
    coefs_df=coefs_df,
    obs_idx=obs_idx_plot,
    a_obs=a_obs_plot,
    P_obs=P_obs_plot,
    title=(
        f"Log-normal EKF — RTS smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure ON]"
        if USE_DISCLOSURE
        else f"Log-normal EKF — RTS smoothed λ_t = exp({EXPONENT}·a_t)  [disclosure OFF]"
    ),
)
plt.show()


In [ ]:
# Flatten (chain, draw) → samples for predictive sampling.
mu_samples      = posterior["mu"].stack(sample=("chain", "draw")).transpose("sample", "time").values
sigma_h_samples = posterior["sigma_h"].stack(sample=("chain", "draw")).values

eps_samples = np.random.normal(
    0,
    sigma_h_samples[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)

n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')
